In [ ]:
# ==============================================================================
# PHASE 4 SETUP: FILE ORGANIZER & SOURCE PACKAGE GENERATOR (Option 2)
# ==============================================================================
import os
import shutil

# Enable automatic reloading of modified modules in src/
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

# Determine base folder location
if os.path.basename(os.getcwd()) == 'traffic_project':
    BASE_DIR = '.'
elif os.path.exists('./traffic_project'):
    BASE_DIR = './traffic_project'
else:
    BASE_DIR = '.'

print(f"📂 Base project directory: {os.path.abspath(BASE_DIR)}\n")

# 1. Define Subdirectories
DIRS = {
    'raw_data':       os.path.join(BASE_DIR, 'raw_data'),
    'processed_data': os.path.join(BASE_DIR, 'processed_data'),
    'checkpoints':    os.path.join(BASE_DIR, 'checkpoints'),
    'results':        os.path.join(BASE_DIR, 'results'),
    'src':            os.path.join(BASE_DIR, 'src'),
}

for folder in DIRS.values():
    os.makedirs(folder, exist_ok=True)

# 2. File Organization Mapping
FILE_MAP = {
    # Raw Data
    'pems-bay.h5': 'raw_data',
    'pems-bay-meta.h5': 'raw_data',
    'adj_mx_bay.pkl': 'raw_data',
    'US_Accidents_March23.csv': 'raw_data',
    'us_accidents_bay_2017.csv': 'raw_data',

    # Processed Data
    'merged_dataset_full.parquet': 'processed_data',
    'data_3d.npy': 'processed_data',
    'adj_tensor.pt': 'processed_data',
    'scaler.pkl': 'processed_data',
    'config.json': 'processed_data',
    'sensors_list.json': 'processed_data',
    'timestamps_list.json': 'processed_data',
    'weather_map.json': 'processed_data',

    # Model Checkpoints
    'lstm_best.pt': 'checkpoints',
    'stgt_best.pt': 'checkpoints',

    # Results & Evaluation Outputs
    'ha_table.npy': 'results',
    'baseline_results.json': 'results',
    'lstm_test_preds.npy': 'results',
    'lstm_test_trues.npy': 'results',
    'stgt_test_preds.npy': 'results',
    'stgt_test_trues.npy': 'results',
    'all_results.json': 'results',
}

# Move existing root files into subdirectories
moved_count = 0
for filename, folder_key in FILE_MAP.items():
    src_path  = os.path.join(BASE_DIR, filename)
    dest_path = os.path.join(DIRS[folder_key], filename)

    if os.path.exists(src_path):
        shutil.move(src_path, dest_path)
        print(f"  ➡️ Moved '{filename}' → {folder_key}/")
        moved_count += 1

print(f"\n✅ File organization complete ({moved_count} files moved)")

# 3. Write Modular Source Package (src/)
SRC_DIR = DIRS['src']

# Write src/__init__.py
with open(os.path.join(SRC_DIR, '__init__.py'), 'w') as f:
    f.write("# Source package initialization\n")

# Write src/dataset.py
dataset_code = '''import torch
import numpy as np
from torch.utils.data import Dataset

class TrafficDataset(Dataset):
    """
    Sliding-window Dataset for 3D traffic tensor data (T, N, F).
    """
    def __init__(self, data_3d, indices, input_steps=12, pred_steps=12, max_samples=None, speed_idx=0):
        self.data        = data_3d
        self.input_steps = input_steps
        self.pred_steps  = pred_steps
        self.speed_idx   = speed_idx

        if max_samples and len(indices) > max_samples:
            idx = np.random.choice(len(indices), max_samples, replace=False)
            self.indices = [indices[i] for i in idx]
        else:
            self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t = self.indices[i]
        x = self.data[t - self.input_steps : t]
        y = self.data[t : t + self.pred_steps, :, self.speed_idx]
        return (torch.tensor(x, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32))
'''
with open(os.path.join(SRC_DIR, 'dataset.py'), 'w') as f:
    f.write(dataset_code)

# Write src/utils.py
utils_code = '''import torch
import numpy as np

def normalize_adjacency_directed(adj):
    """Row-normalize directed graph adjacency matrix so rows sum to 1."""
    row_sum = adj.sum(dim=1, keepdim=True).clamp(min=1e-8)
    return adj / row_sum

def inverse_scale_speed(y_scaled, scaler, speed_col_idx=0):
    """Convert normalized z-scores back to physical miles per hour (mph)."""
    mean = scaler.mean_[speed_col_idx]
    std = scaler.scale_[speed_col_idx]
    return y_scaled * std + mean

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mape(y_true, y_pred, eps=1e-5):
    mask = np.abs(y_true) > eps
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
'''
with open(os.path.join(SRC_DIR, 'utils.py'), 'w') as f:
    f.write(utils_code)

# Write src/models.py
models_code = '''import torch
import torch.nn as nn
import torch.nn.functional as F

class DiffusionConvLayer(nn.Module):
    """Bidirectional Diffusion Graph Convolution."""
    def __init__(self, d_model, n_hops=2):
        super().__init__()
        self.n_hops = n_hops
        self.fwd_linears = nn.ModuleList([
            nn.Linear(d_model, d_model, bias=False) for _ in range(n_hops)
        ])
        self.bwd_linears = nn.ModuleList([
            nn.Linear(d_model, d_model, bias=False) for _ in range(n_hops)
        ])
        self.out = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, adj_fwd, adj_bwd):
        h_fwd = x
        for layer in self.fwd_linears:
            h_fwd = F.relu(layer(torch.einsum('nm, bmd -> bnd', adj_fwd, h_fwd)))

        h_bwd = x
        for layer in self.bwd_linears:
            h_bwd = F.relu(layer(torch.einsum('nm, bmd -> bnd', adj_bwd, h_bwd)))

        combined = torch.cat([h_fwd, h_bwd], dim=-1)
        out = self.out(combined)
        return self.norm(out + x)


class SpatioTemporalTransformer(nn.Module):
    """Spatial-Temporal Transformer architecture combining GNN and Transformer Encoder."""
    def __init__(self, n_features, d_model, n_heads, n_gnn_layers, n_tf_layers, n_sensors, pred_steps, input_steps=12, dropout=0.1):
        super().__init__()
        self.n_sensors   = n_sensors
        self.pred_steps  = pred_steps
        self.input_steps = input_steps
        self.d_model     = d_model

        self.input_proj = nn.Linear(n_features, d_model)
        self.gnn_layers = nn.ModuleList([
            DiffusionConvLayer(d_model, n_hops=2) for _ in range(n_gnn_layers)
        ])
        self.spatial_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )
        self.pos_embedding = nn.Parameter(torch.randn(1, input_steps, d_model) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_tf_layers)
        self.tf_norm = nn.LayerNorm(d_model)

        self.pred_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, pred_steps)
        )

    def forward(self, x, adj_fwd, adj_bwd):
        B, T, N, n_feat = x.shape

        x = self.input_proj(x)
        x_orig = x.clone()

        x_flat = x.reshape(B * T, N, self.d_model)
        for gnn in self.gnn_layers:
            x_flat = gnn(x_flat, adj_fwd, adj_bwd)

        x_spatial = x_flat.reshape(B, T, N, self.d_model)

        gate_input = torch.cat([x_spatial, x_orig], dim=-1)
        gate       = self.spatial_gate(gate_input)
        x          = gate * x_spatial + (1 - gate) * x_orig

        x = x.permute(0, 2, 1, 3).reshape(B * N, T, self.d_model)
        x = x + self.pos_embedding[:, :T, :]
        x = self.transformer(x)
        x = self.tf_norm(x[:, -1, :])

        pred = self.pred_head(x)
        pred = pred.reshape(B, N, self.pred_steps)
        pred = pred.permute(0, 2, 1)
        return pred
'''
with open(os.path.join(SRC_DIR, 'models.py'), 'w') as f:
    f.write(models_code)

print("✅ Modular source files ready in 'src/':")
print("  ├── src/__init__.py")
print("  ├── src/dataset.py")
print("  ├── src/utils.py")
print("  └── src/models.py")

# Define path shortcuts for Phase 4 code cells
PROCESSED_DIR = DIRS['processed_data']
CHECKPOINT_DIR = DIRS['checkpoints']
RESULTS_DIR    = DIRS['results']